In [5]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from linearmodels.panel import PanelOLS

In [2]:
data = pd.read_csv("ELA_data_2013_2023.csv")
school_poverty = pd.read_excel('Demographic_Snapshot_2017-18_to_2021-22__Public_.xlsx', sheet_name = 'School')

from data_cleaning import cleaning_data, change_variable_type, create_school_level_data, merge_data_and_poverty, categorize_poverty, categorize_title_i, categorize_economic_need
# clean data
school_data = cleaning_data(data) 

# convert num columns
cols = ['mean_scale_score','level_1_count','level_2_count','level_3_count', 'level_4_count', 'level_4_percentage', 'level_3_4_count',
        'level_1_percentage','level_2_percentage','level_3_percentage','level_4_percentage','level_3_4_percentage']
school_data = change_variable_type(school_data,cols)
# change year to object
school_data['Year'] = school_data['Year'].astype('object')


# create school level data, all grades all students
school_level_data = create_school_level_data(school_data)

# merge with poverty data
merged_data = merge_data_and_poverty(school_level_data, school_poverty)

# convert poverty percentage to float and categorize, round to 3 decimals
merged_data['% Poverty'] = merged_data['% Poverty'].astype(float)
merged_data['Economic Need Index'] = merged_data['Economic Need Index'].astype(float)

merged_data['Poverty_Category'] = merged_data.apply(categorize_poverty, axis=1)
merged_data['Economic_Need_Index_Category'] = merged_data.apply(categorize_economic_need, axis=1)

merged_data['% Poverty'] = merged_data['% Poverty'].round(3)
merged_data['Economic Need Index'] = merged_data['Economic Need Index'].round(3)

#### Data split by grades

In [18]:
# GRADE 3
school_data_3 = school_data[
    (school_data['Grade'] == '3') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

merged_data3 = merge_data_and_poverty(school_data_3, school_poverty)

# fix poverty percentage to float, categorize, round to 3 decimals
merged_data3['% Poverty'] = merged_data3['% Poverty'].astype(float)
merged_data3['Poverty_Category'] = merged_data3.apply(categorize_poverty, axis=1)
merged_data3['% Poverty'] = merged_data3['% Poverty'].round(3)

# set index for panel data, filter for schools with 2 years of data (2019 and 2022)
data_grade3= merged_data3.set_index(['school_name','Year'])

counts3 = data_grade3.groupby(level=0).size() # check count of years for each school
valid_schools3 = counts3[counts3 == 2].index # filter data to include only valid schools
data_grade3_adjust = data_grade3.loc[valid_schools3]

data_grade3_adjust['Poverty_Category'] = data_grade3_adjust['Poverty_Category'].astype(int)

# vars for did model
data_grade3_adjust['treated'] = (data_grade3_adjust['Poverty_Category'] == 1).astype(int)
data_grade3_adjust['post'] = (data_grade3_adjust.index.get_level_values('Year') == 2022).astype(int)
data_grade3_adjust['did'] = (data_grade3_adjust['treated'] * data_grade3_adjust['post']).astype(int)

# GRADE 4
school_data_4 = school_data[
    (school_data['Grade'] == '4') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

merged_data4 = merge_data_and_poverty(school_data_4, school_poverty)

# fix poverty percentage to float, categorize, round to 3 decimals
merged_data4['% Poverty'] = merged_data4['% Poverty'].astype(float)
merged_data4['Poverty_Category'] = merged_data4.apply(categorize_poverty, axis=1)
merged_data4['% Poverty'] = merged_data4['% Poverty'].round(3)

# set index for panel data, filter for schools with 2 years of data (2019 and 2022)
data_grade4= merged_data4.set_index(['school_name','Year'])

counts4 = data_grade4.groupby(level=0).size() # check count of years for each school
valid_schools4 = counts4[counts4 == 2].index # filter data to include only valid schools
data_grade4_adjust = data_grade4.loc[valid_schools4]

data_grade4_adjust['Poverty_Category'] = data_grade4_adjust['Poverty_Category'].astype(int)

# vars for did model
data_grade4_adjust['treated'] = (data_grade4_adjust['Poverty_Category'] == 1).astype(int)
data_grade4_adjust['post'] = (data_grade4_adjust.index.get_level_values('Year') == 2022).astype(int)
data_grade4_adjust['did'] = (data_grade4_adjust['treated'] * data_grade4_adjust['post']).astype(int)


# GRADE 5
school_data_5 = school_data[
    (school_data['Grade'] == '5') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

merged_data5 = merge_data_and_poverty(school_data_5, school_poverty)

merged_data5['% Poverty'] = merged_data5['% Poverty'].astype(float)
merged_data5['Poverty_Category'] = merged_data5.apply(categorize_poverty, axis=1)
merged_data5['% Poverty'] = merged_data5['% Poverty'].round(3)

data_grade5 = merged_data5.set_index(['school_name','Year'])

counts5 = data_grade5.groupby(level=0).size()
valid_schools5 = counts5[counts5 == 2].index
data_grade5_adjust = data_grade5.loc[valid_schools5]

data_grade5_adjust['Poverty_Category'] = data_grade5_adjust['Poverty_Category'].astype(int)

data_grade5_adjust['treated'] = (data_grade5_adjust['Poverty_Category'] == 1).astype(int)
data_grade5_adjust['post'] = (data_grade5_adjust.index.get_level_values('Year') == 2022).astype(int)
data_grade5_adjust['did'] = (data_grade5_adjust['treated'] * data_grade5_adjust['post']).astype(int)


# GRADE 6
school_data_6 = school_data[
    (school_data['Grade'] == '6') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

merged_data6 = merge_data_and_poverty(school_data_6, school_poverty)

merged_data6['% Poverty'] = merged_data6['% Poverty'].astype(float)
merged_data6['Poverty_Category'] = merged_data6.apply(categorize_poverty, axis=1)
merged_data6['% Poverty'] = merged_data6['% Poverty'].round(3)

data_grade6 = merged_data6.set_index(['school_name','Year'])

counts6 = data_grade6.groupby(level=0).size()
valid_schools6 = counts6[counts6 == 2].index
data_grade6_adjust = data_grade6.loc[valid_schools6]

data_grade6_adjust['Poverty_Category'] = data_grade6_adjust['Poverty_Category'].astype(int)

data_grade6_adjust['treated'] = (data_grade6_adjust['Poverty_Category'] == 1).astype(int)
data_grade6_adjust['post'] = (data_grade6_adjust.index.get_level_values('Year') == 2022).astype(int)
data_grade6_adjust['did'] = (data_grade6_adjust['treated'] * data_grade6_adjust['post']).astype(int)


# GRADE 7
school_data_7 = school_data[
    (school_data['Grade'] == '7') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

merged_data7 = merge_data_and_poverty(school_data_7, school_poverty)

merged_data7['% Poverty'] = merged_data7['% Poverty'].astype(float)
merged_data7['Poverty_Category'] = merged_data7.apply(categorize_poverty, axis=1)
merged_data7['% Poverty'] = merged_data7['% Poverty'].round(3)

data_grade7 = merged_data7.set_index(['school_name','Year'])

counts7 = data_grade7.groupby(level=0).size()
valid_schools7 = counts7[counts7 == 2].index
data_grade7_adjust = data_grade7.loc[valid_schools7]

data_grade7_adjust['Poverty_Category'] = data_grade7_adjust['Poverty_Category'].astype(int)

data_grade7_adjust['treated'] = (data_grade7_adjust['Poverty_Category'] == 1).astype(int)
data_grade7_adjust['post'] = (data_grade7_adjust.index.get_level_values('Year') == 2022).astype(int)
data_grade7_adjust['did'] = (data_grade7_adjust['treated'] * data_grade7_adjust['post']).astype(int)


# GRADE 8
school_data_8 = school_data[
    (school_data['Grade'] == '8') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

merged_data8 = merge_data_and_poverty(school_data_8, school_poverty)

merged_data8['% Poverty'] = merged_data8['% Poverty'].astype(float)
merged_data8['Poverty_Category'] = merged_data8.apply(categorize_poverty, axis=1)
merged_data8['% Poverty'] = merged_data8['% Poverty'].round(3)

data_grade8 = merged_data8.set_index(['school_name','Year'])

counts8 = data_grade8.groupby(level=0).size()
valid_schools8 = counts8[counts8 == 2].index
data_grade8_adjust = data_grade8.loc[valid_schools8]

data_grade8_adjust['Poverty_Category'] = data_grade8_adjust['Poverty_Category'].astype(int)

data_grade8_adjust['treated'] = (data_grade8_adjust['Poverty_Category'] == 1).astype(int)
data_grade8_adjust['post'] = (data_grade8_adjust.index.get_level_values('Year') == 2022).astype(int)
data_grade8_adjust['did'] = (data_grade8_adjust['treated'] * data_grade8_adjust['post']).astype(int)

### Modeling

In [ ]:
# data_grade3_adjust['treated_cont'] = data_grade3_adjust['% Poverty'] * data_grade3_adjust['post']

In [30]:
data_grade3_adjust['poverty_percent'] = data_grade3_adjust['% Poverty'] * 100
data_grade3_adjust['treated_cont'] = data_grade3_adjust['poverty_percent'] * data_grade3_adjust['post']

data_grade4_adjust['poverty_percent'] = data_grade4_adjust['% Poverty'] * 100
data_grade4_adjust['treated_cont'] = data_grade4_adjust['poverty_percent'] * data_grade4_adjust['post']

data_grade5_adjust['poverty_percent'] = data_grade5_adjust['% Poverty'] * 100
data_grade5_adjust['treated_cont'] = data_grade5_adjust['poverty_percent'] * data_grade5_adjust['post']

data_grade6_adjust['poverty_percent'] = data_grade6_adjust['% Poverty'] * 100
data_grade6_adjust['treated_cont'] = data_grade6_adjust['poverty_percent'] * data_grade6_adjust['post']

data_grade7_adjust['poverty_percent'] = data_grade7_adjust['% Poverty'] * 100
data_grade7_adjust['treated_cont'] = data_grade7_adjust['poverty_percent'] * data_grade7_adjust['post']

data_grade8_adjust['poverty_percent'] = data_grade8_adjust['% Poverty'] * 100
data_grade8_adjust['treated_cont'] = data_grade8_adjust['poverty_percent'] * data_grade8_adjust['post']

#### Grade 3

In [31]:
model3 = PanelOLS(
    dependent=data_grade3_adjust["mean_scale_score"],
    exog=data_grade3_adjust[["treated_cont"]],
    entity_effects=True,
    time_effects=True,
    weights=data_grade3_adjust['number_tested']
)

res3 = model3.fit(cov_type='clustered', cluster_entity=True)

print(res3.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0383
Estimator:                   PanelOLS   R-squared (Between):             -0.0047
No. Observations:                1536   R-squared (Within):              -0.0655
Date:                Thu, Mar 26 2026   R-squared (Overall):             -0.0047
Time:                        14:03:15   Log-likelihood                   -3603.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      30.352
Entities:                         771   P-value                           0.0000
Avg Obs:                       1.9922   Distribution:                   F(1,763)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             15.194
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Grade 4

In [32]:
model4 = PanelOLS(
    dependent=data_grade4_adjust["mean_scale_score"],
    exog=data_grade4_adjust[["treated_cont"]],
    entity_effects=True,
    time_effects=True,
    weights=data_grade4_adjust['number_tested']
)

res4 = model4.fit(cov_type='clustered', cluster_entity=True)

print(res4.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0694
Estimator:                   PanelOLS   R-squared (Between):             -0.0072
No. Observations:                1515   R-squared (Within):               0.1967
Date:                Thu, Mar 26 2026   R-squared (Overall):             -0.0072
Time:                        14:03:50   Log-likelihood                   -3666.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      56.038
Entities:                         761   P-value                           0.0000
Avg Obs:                       1.9908   Distribution:                   F(1,752)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             29.018
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Grade 5

In [33]:
model5 = PanelOLS(
    dependent=data_grade5_adjust["mean_scale_score"],
    exog=data_grade5_adjust[["treated_cont"]],
    entity_effects=True,
    time_effects=True,
    weights=data_grade5_adjust['number_tested']
)

res5 = model5.fit(cov_type='clustered', cluster_entity=True)

print(res5.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0090
Estimator:                   PanelOLS   R-squared (Between):              0.0022
No. Observations:                1507   R-squared (Within):               0.0379
Date:                Thu, Mar 26 2026   R-squared (Overall):              0.0022
Time:                        14:04:32   Log-likelihood                   -3450.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      6.8440
Entities:                         755   P-value                           0.0091
Avg Obs:                       1.9960   Distribution:                   F(1,750)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             4.0616
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Grade 6

In [34]:
model6 = PanelOLS(
    dependent=data_grade6_adjust["mean_scale_score"],
    exog=data_grade6_adjust[["treated_cont"]],
    entity_effects=True,
    time_effects=True,
    weights=data_grade6_adjust['number_tested']
)

res6 = model6.fit(cov_type='clustered', cluster_entity=True)

print(res6.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.1504
Estimator:                   PanelOLS   R-squared (Between):              0.0110
No. Observations:                 942   R-squared (Within):               0.0132
Date:                Thu, Mar 26 2026   R-squared (Overall):              0.0110
Time:                        14:05:10   Log-likelihood                   -2126.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      82.817
Entities:                         472   P-value                           0.0000
Avg Obs:                       1.9958   Distribution:                   F(1,468)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             25.882
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Grade 7

In [35]:
model7 = PanelOLS(
    dependent=data_grade7_adjust["mean_scale_score"],
    exog=data_grade7_adjust[["treated_cont"]],
    entity_effects=True,
    time_effects=True,
    weights=data_grade7_adjust['number_tested']
)

res7 = model7.fit(cov_type='clustered', cluster_entity=True)

print(res7.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.1556
Estimator:                   PanelOLS   R-squared (Between):              0.0102
No. Observations:                 924   R-squared (Within):               0.4634
Date:                Thu, Mar 26 2026   R-squared (Overall):              0.0102
Time:                        14:05:41   Log-likelihood                   -1954.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      84.567
Entities:                         463   P-value                           0.0000
Avg Obs:                       1.9957   Distribution:                   F(1,459)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             18.473
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Grade 8

In [36]:
model8 = PanelOLS(
    dependent=data_grade8_adjust["mean_scale_score"],
    exog=data_grade8_adjust[["treated_cont"]],
    entity_effects=True,
    time_effects=True,
    weights=data_grade8_adjust['number_tested']
)

res8 = model8.fit(cov_type='clustered', cluster_entity=True)

print(res8.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0005
Estimator:                   PanelOLS   R-squared (Between):              0.0005
No. Observations:                 917   R-squared (Within):              -0.0020
Date:                Thu, Mar 26 2026   R-squared (Overall):              0.0005
Time:                        14:06:08   Log-likelihood                   -1826.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.2185
Entities:                         460   P-value                           0.6404
Avg Obs:                       1.9935   Distribution:                   F(1,455)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             0.0582
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
